[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/02-data-harmonization/04-combining_mappings_and_aliases_in_one_index.ipynb)

# Combining Character Mappings and Alias Sets

Inventory and product data has a habit of accumulating two unrelated problems on the exact same field:

- **OCR / scanning noise**,  a barcode or scanned inventory sheet misreads `"B88-EXT"` as `"8B8-3XT"`, confusing visually similar characters. This is a spelling-level problem, solved by a custom `CharacterMapping`.
- **Legacy identifiers**,  an older system, or a warehouse team that hasn't updated their labels, still refers to the same product by a discontinued internal code, like `"LGCY-441"`. This is a vocabulary-level problem, solved by an `AliasSet`.

Neither tool fixes the other's problem. A product ID field harmonized with only one of them still fails on the other. In this notebook, you'll attach both to the same field at once.

In this notebook you will:

1. See two independent queries against the same field fail for two different reasons
2. Build a custom `CharacterMapping` for OCR-style character confusion,  no umlaut handling needed, this is plain English/alphanumeric data
3. Build an `AliasSet` mapping legacy product codes to their current IDs
4. Attach both to the same field in one index and resolve both queries
5. Learn the one rule for keeping the two techniques from working against each other

**Estimated time:** 10 minutes
**Difficulty:** Intermediate
**Prerequisites:** `character_mapping.ipynb`, `alias_sets_nicknames_and_synonyms.ipynb`, `building_a_custom_character_mapping.ipynb`


## Install the SDK

M|BOX is distributed on PyPI. Run the cell below to install it (or run this in your terminal without the `!`).

In [1]:
# !pip install mbox

## 1. Two different failures, same field

Here's a small product catalog, loaded from `datasets/product_catalog.csv`.

In [2]:
import pandas as pd
from mbox.indexing import TableIndexer

df = pd.read_csv("datasets/product_catalog.csv")

df

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,product_id,product_name
0,B88-EXT,Extended Battery Pack
1,A12-PWR,Portable Power Bank
2,C99-SNS,Motion Sensor Camera
3,D45-REL,Smart Relay Switch


**Query A,  OCR noise.** A warehouse scanner misreads `"B88-EXT"` as `"8B8-3XT"` (`B`/`8` and `E`/`3` confusion).

**Query B,  a legacy code.** A partner system still references this same product by its old internal code, `"LGCY-441"`, which has nothing in common, character-for-character, with `"B88-EXT"`.

Let's run both against a plain index with no harmonization at all.

In [3]:
baseline_index = TableIndexer.create_index(df, index_columns=["product_id"], tmp_dir="tmp_index")

query_a = baseline_index.match(product_id="8B8-3XT", include_field_scores=True)
query_b = baseline_index.match(product_id="LGCY-441", include_field_scores=True)
 
print("Query A (OCR noise):")
display(query_a)

print("\nQuery B (legacy code):")
display(query_b)

Query A (OCR noise):


,query_row,index_row,product_id_candidate,product_name_candidate,overall_score,product_id_score
0,0,0,B88-EXT,Extended Battery Pack,36,36



Query B (legacy code):


,query_row,index_row,product_id_candidate,product_name_candidate,overall_score,product_id_score
0,0,-1,,,0,0


Query A fails because raw character comparison doesn't know `8` and `B` are meant to be equivalent here. Query B fails because `"LGCY-441"` isn't a garbled version of `"B88-EXT"` at all,  it's a different, valid identifier for the same product that the engine has simply never been told about. A character mapping can't fix Query B. An alias can't fix Query A. Both need to be solved, and neither tool alone gets you there.

## 2. A custom `CharacterMapping` for OCR noise

This is plain English alphanumeric data,  no umlauts, no accents, so there's nothing for `expand_umlauts` or `create_german_standard()` to do here. What we need instead is `numbers_as_characters`, which normalizes common leetspeak-style digit/letter confusion, combined with a `mapped_characters` set that keeps the hyphen (a structural character in this ID format).

In [4]:
from mbox.mapping import CharacterMapping

product_id_mapping = CharacterMapping(
    name="product_id_ocr_cleaner",
    mapped_characters="A-Z0-9-",
    map_upper=True,
    deaccentuate=True,
    expand_umlauts=False,  
    numbers_as_characters=True
)

print(product_id_mapping.apply("B88-EXT"))
print(product_id_mapping.apply("8B8-3XT"))

BBB-EXT
BBB-EXT


Both should normalize to the same string. That resolves Query A's problem,  but Query B's legacy code still doesn't share enough characters with `"B88-EXT"` for any amount of normalization to bridge the gap.

## 3. An `AliasSet` for legacy product codes

For the vocabulary problem, define each legacy code as an alias pointing at its current `product_id`.

In [5]:
from mbox.aliases import AliasSet

legacy_code_aliases = AliasSet(name="legacy_product_codes")
legacy_code_aliases.add(word="B88-EXT", alias="LGCY-441", penalty=0)
legacy_code_aliases.add(word="A12-PWR", alias="LGCY-198", penalty=0)

AliasSet(name='legacy_product_codes', aliases=OrderedDict({'{"word":"B88-EXT","alias":"LGCY-441"}': 0, '{"word":"A12-PWR","alias":"LGCY-198"}': 0}))

## 4. Attaching both to the same field

Now build the index with `character_mappings` and `alias_sets` both attached to `product_id`, and re-run Query A and Query B.

In [6]:
harmonized_index = TableIndexer.create_index(
    df=df,
    index_columns=["product_id"],
    character_mappings={"product_id": product_id_mapping},
    alias_sets={"product_id": legacy_code_aliases},
    tmp_dir="tmp_index"
)

query_a_harmonized = harmonized_index.match(product_id="8B8-3XT", include_field_scores=True)
query_b_harmonized = harmonized_index.match(product_id="LGCY-441", include_field_scores=True)

print("Query A (OCR noise), harmonized:")
display(query_a_harmonized)

print("\nQuery B (legacy code), harmonized:")
display(query_b_harmonized)

Query A (OCR noise), harmonized:


,query_row,index_row,product_id_candidate,product_name_candidate,overall_score,product_id_score
0,0,0,B88-EXT,Extended Battery Pack,100,100



Query B (legacy code), harmonized:


,query_row,index_row,product_id_candidate,product_name_candidate,overall_score,product_id_score
0,0,0,B88-EXT,Extended Battery Pack,100,100


Compare both results against the baseline in Step 1. Query A should now score well because of the character mapping; Query B should now score well because of the alias,  and both are handled by a single index, built with a single `create_index()` call. Neither tool had to know about the other's job.

## 5. The one rule: define aliases using the real indexed spelling

Your `CharacterMapping` normalizes text,  including the *result* of resolving an alias,  before comparison happens. If an alias's `word` doesn't exactly match the real value stored in your data, you can end up with a subtle mismatch instead of a clean one.

```python
# Risky: doesn't exactly match the real indexed product_id
inconsistent_aliases = AliasSet(name="risky_example")
inconsistent_aliases.add(word="b88ext", alias="LGCY-441", penalty=0)   # missing the hyphen
```

Here, the indexed value is `"B88-EXT"`,  with a hyphen, which our `mapped_characters` setting explicitly preserves. But the alias's `word` was written as `"b88ext"`, with no hyphen at all. After normalization, the two may no longer line up exactly, and the alias could silently fail to resolve.

**The safe pattern:** define `word` using the exact spelling that appears in your DataFrame,  hyphen and all,  and let the `CharacterMapping` handle normalizing both sides consistently:

```python
# Safe: matches the real indexed value exactly
safe_aliases = AliasSet(name="safe_example")
safe_aliases.add(word="B88-EXT", alias="LGCY-441", penalty=0)
```

This is exactly what Step 3 did, which is why it worked cleanly.

## Next steps

You've now used every core data harmonization tool M|BOX provides, individually and together. From here:

- **`03-index-configuration/`**,  attach both `character_mapping` and `aliases` explicitly through `TableFieldConfig`, as part of a versioned, declarative schema rather than inline dictionaries
- **`04-recall-tuning/`**,  once your data is harmonized, control exactly how much each field contributes to the overall match score

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*